In [9]:
import cv2
import numpy as np
import time
import gradio as gr
import onnxruntime as ort

# -----------------------------
# Load ONNX model (CPU only)
# -----------------------------
session = ort.InferenceSession("models/fight_detection_model.onnx", providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# -----------------------------
# Global control flag
# -----------------------------
running = False

# -----------------------------
# Webcam + Violence Detection
# -----------------------------
def webcam_stream():
    global running
    running = True

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam. Check your camera.")

    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))

    # Video writer for saving fights
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('output.mp4', fourcc, 20, (frame_width, frame_height))

    sliding_window = []
    max_frames = 64  # Match training frames
    DETECTION_INTERVAL = 10
    last_detection_time = time.time()
    is_detecting = False
    recording = False

    while running:
        ret, frame = cap.read()
        if not ret:
            break

        current_time = time.time()

        # Start detection every 10 sec
        if not is_detecting and (current_time - last_detection_time >= DETECTION_INTERVAL):
            is_detecting = True

        if is_detecting or recording:
            resized = cv2.resize(frame, (224, 224)) / 255.0
            sliding_window.append(resized)

            if len(sliding_window) == max_frames:
                input_frames = np.expand_dims(np.array(sliding_window), axis=0).astype('float32')

                # ONNX inference
                prediction = session.run([output_name], {input_name: input_frames})[0]
                predicted_class = np.argmax(prediction, axis=1)[0]

                if predicted_class == 1:  # Violence detected
                    recording = True
                    # Draw FIGHT on frame
                    cv2.putText(frame, "FIGHT", (50, frame_height - 50),
                                cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
                    out.write(frame)  # Save frame to video
                else:
                    if recording:
                        recording = False
                    is_detecting = False
                    last_detection_time = current_time

                sliding_window.pop(0)

        # Convert BGR → RGB for Gradio
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        yield frame_rgb

    cap.release()
    out.release()


# -----------------------------
# Stop function
# -----------------------------
def stop_camera():
    global running
    running = False
    return "Stopped"


# -----------------------------
# Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("# 🎥 Violence Detection System")

    with gr.Row():
        start_btn = gr.Button("▶️ Start Camera")
        stop_btn = gr.Button("⏹ Stop")

    live_feed = gr.Image(label="Live Feed")

    # Connect buttons
    start_btn.click(fn=webcam_stream, outputs=live_feed)
    stop_btn.click(fn=stop_camera)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
